# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hisham-Walid/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

My provisional choice is **Lane 4: CTR / Engagement Opportunity Scoring**. I want to identify visible pages that appear to under-capture clicks relative to comparable pages at a similar search-position tier. This lane extends the strongest Week 1 observation: CTR changes sharply with position, so a single global CTR threshold would mix together pages facing very different search-result conditions. The lane is provisional through Week 4, but it offers a useful, explainable decision-support problem with enough starter data to test before using the larger warehouse release.

In [1]:
LANE = "CTR / Engagement Opportunity Scoring"
DECISION_GRAIN = "one pseudonymized content page"
print(f"Provisional lane: {LANE}")
print(f"Decision grain: {DECISION_GRAIN}")


Provisional lane: CTR / Engagement Opportunity Scoring
Decision grain: one pseudonymized content page


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which visible content pages under-capture clicks relative to pages in the same position tier, and therefore deserve review first?

- **Unit of analysis:** one pseudonymized content page.
- **Output:** a ranked review queue with an opportunity score, confidence label, and transparent reason codes.
- **Decision and action:** a content strategist or editor decides which limited set of pages to inspect first, then may review title/meta wording, intent match, snippet structure, or on-page engagement. The score recommends *review*, not an automatic rewrite.
- **Cost of a wrong call:** a false positive wastes scarce editorial time and risks unnecessary changes; a false negative misses a page that may be losing clicks despite useful visibility. Because review capacity is limited, **Precision@20** is the provisional primary metric.
- **Why data/ML may help:** position, impressions, content type, age, freshness, and engagement interact in ways a single threshold cannot represent fairly. I will still build a transparent position-adjusted rule first; a learned model earns a place only if honest validation shows useful improvement over that baseline.

In [2]:
MIN_VISIBLE_IMPRESSIONS = 100
REVIEW_SCREEN_IMPRESSIONS = 500
MAX_REVIEW_POSITION = 20
LOW_CTR_PCT = 0.5  # Rate columns are stored on a 0-100 percentage scale.
REVIEW_CAPACITY = 20
print({
    "minimum_visible_impressions": MIN_VISIBLE_IMPRESSIONS,
    "review_capacity": REVIEW_CAPACITY,
    "primary_metric": f"Precision@{REVIEW_CAPACITY}",
})


{'minimum_visible_impressions': 100, 'review_capacity': 20, 'primary_metric': 'Precision@20'}


## 3. Quick look at the data (2-3 real numbers)

The starter slice is large enough to compare position-adjusted CTR groups and contains a substantial pool of visible, low-CTR pages. The screening rule below is **not** the final model or a claim that every flagged page needs editing; it is an explainable first count that shows why a ranked review queue could be useful.

In [3]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
while root != root.parent and not (root / "data/raw/content_refresh_anonymized.csv").exists():
    root = root.parent
data_path = root / "data/raw/content_refresh_anonymized.csv"
assert data_path.exists(), "Run this notebook from inside the starter repository."

df = pd.read_csv(data_path)
required = {"content_id", "client_id", "impressions_90d", "avg_position", "ctr", "position_tier"}
assert required.issubset(df.columns)

visible = df[(df["impressions_90d"] >= MIN_VISIBLE_IMPRESSIONS) & (df["avg_position"] > 0)].copy()
review_screen = visible[
    (visible["impressions_90d"] >= REVIEW_SCREEN_IMPRESSIONS)
    & (visible["avg_position"] <= MAX_REVIEW_POSITION)
    & (visible["ctr"] < LOW_CTR_PCT)
].copy()

print(f"1) Starter data: {len(df):,} pages across {df['client_id'].nunique()} pseudonymized clients.")
print(f"2) Visible comparison pool: {len(visible):,} pages ({len(visible) / len(df):.1%} of the slice).")
print(f"3) Simple review screen: {len(review_screen):,} pages representing "
      f"{review_screen['impressions_90d'].sum():,.0f} observed impressions.")

tier_summary = (
    visible.groupby("position_tier", observed=True)
    .agg(pages=("content_id", "size"), mean_ctr_pct=("ctr", "mean"), median_ctr_pct=("ctr", "median"))
    .sort_values("mean_ctr_pct", ascending=False)
)
display(tier_summary.style.format({"mean_ctr_pct": "{:.3f}", "median_ctr_pct": "{:.3f}"}))


1) Starter data: 30,000 pages across 32 pseudonymized clients.
2) Visible comparison pool: 22,006 pages (73.4% of the slice).
3) Simple review screen: 9,759 pages representing 90,968,008 observed impressions.


,pages,mean_ctr_pct,median_ctr_pct
position_tier,,,
page_1,8633,0.355,0.230
top_3,533,0.334,0.190
striking,5903,0.256,0.150
page_3_5,6058,0.142,0.060
deep,879,0.055,0.000


## 4. Careful words: what I can and can't claim

This work can report **observed, measured associations** and provide **directional decision support** about which pages merit human review. A low position-adjusted CTR is an opportunity signal, not proof that metadata or content is defective. The project will not claim that it discovered Google's ranking algorithm, that a recommendation will increase traffic, or that a refresh caused recovery. Those causal claims would require intervention or experimental evidence that this observational dataset does not provide. Later validation will keep clients separated and, when the warehouse time series is used, keep feature windows strictly before outcome windows.

In [4]:
planned_features = {"impressions_90d", "avg_position", "ctr", "content_type", "content_age_days", "days_since_last_update"}
leakage_or_identity_fields = {"trend_pct", "trend_direction", "content_id", "client_id"}
assert planned_features.isdisjoint(leakage_or_identity_fields)
print("Safety check passed: no identifiers or current trend-derived label fields are planned as model features.")


Safety check passed: no identifiers or current trend-derived label fields are planned as model features.


## Self-check

Before submitting, I verified:

- [x] Every section above is filled with written reasoning and supporting code
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries are displayed
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] The completed notebook is under `work/notebooks/` and ready to commit